# TrainAiImport — Colab pipeline
Huấn luyện trên Colab GPU; dữ liệu đặt dưới `MyDrive/eform_btp/Module/AI Import/Data`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, random, re, hashlib
from collections import Counter

MYDRIVE = Path('/content/drive/MyDrive')
DATA_ROOT = MYDRIVE / 'eform_btp/Module/AI Import/Data'
RAW_DIR = DATA_ROOT / 'Raw'
LABEL_DIR = DATA_ROOT / 'Labels'
TRAIN_DIR = DATA_ROOT / 'Train'
VAL_DIR = DATA_ROOT / 'Validation'
TEST_DIR = DATA_ROOT / 'Test'
for p in [LABEL_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR]: p.mkdir(parents=True, exist_ok=True)
print(DATA_ROOT, DATA_ROOT.exists())

In [ ]:
%pip -q install pandas openpyxl rapidfuzz tqdm sentence-transformers
import pandas as pd
from openpyxl import load_workbook
from tqdm.auto import tqdm
print('dependencies ok')

In [ ]:
def read_jsonl(path):
    if not path.exists(): return []
    return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in rows), encoding='utf-8')

DOCTYPE_RE = re.compile(r'_(\d{2}[a-z]?)_', re.I)
def doctype(path):
    m=DOCTYPE_RE.search(path.stem); return m.group(1).lower() if m else ''

raw_files=sorted(p for p in RAW_DIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.xlsx','.xlsm'} and not p.name.startswith('~$'))
print('raw workbooks:', len(raw_files))
print('doctype:', sorted({doctype(p) for p in raw_files if doctype(p)}))

In [ ]:
# Làm sạch mapping candidates, loại record thiếu định danh hoặc trùng
source = read_jsonl(LABEL_DIR/'mappings.jsonl')
clean=[]; seen=set(); dropped=Counter()
for r in source:
    key=(r.get('file',''), r.get('sheet',''), r.get('excel_column',''), r.get('column_code',''))
    if not r.get('file') or not r.get('header_path') or not r.get('column_code'): dropped['missing']+=1; continue
    if key in seen: dropped['duplicate']+=1; continue
    seen.add(key); r['doc_type_code']=doctype(RAW_DIR/r['file']) or r.get('doc_type_code',''); clean.append(r)
write_jsonl(LABEL_DIR/'mappings_clean.jsonl', clean)
print('input:',len(source),'clean:',len(clean),'dropped:',dict(dropped))
print(pd.Series([r['doc_type_code'] for r in clean]).value_counts().head(30))

In [ ]:
# Chỉ record verified mới được dùng train; chia theo workbook chống leakage
verified=[r for r in clean if r.get('verified') and r.get('target_data_field_id')]
files=sorted({r['file'] for r in verified}); random.Random(42).shuffle(files)
n=len(files); n_test=max(1,round(n*.15)) if n else 0; n_val=max(1,round(n*.15)) if n>2 else 0
test=set(files[:n_test]); val=set(files[n_test:n_test+n_val]); train=set(files[n_test+n_val:])
train_rows=[r for r in verified if r['file'] in train]; val_rows=[r for r in verified if r['file'] in val]; test_rows=[r for r in verified if r['file'] in test]
write_jsonl(TRAIN_DIR/'train.jsonl',train_rows); write_jsonl(VAL_DIR/'validation.jsonl',val_rows); write_jsonl(TEST_DIR/'test.jsonl',test_rows)
print({'verified':len(verified),'workbooks':n,'train':len(train_rows),'validation':len(val_rows),'test':len(test_rows)})

In [ ]:
# Kiểm tra test và đối chiếu một số file raw
assert not (train & val or train & test or val & test)
for row in random.Random(42).sample(test_rows, min(5,len(test_rows))):
    path=RAW_DIR/row['file']; assert path.exists(), path
    wb=load_workbook(path, read_only=True, data_only=False)
    print({'file':path.name,'sheet':row['sheet'],'column':row['excel_column'],'doctype':doctype(path),'target':row['target_data_field_id'],'sheets':wb.sheetnames[:5]})
    wb.close()
print('test verification passed')

In [ ]:
# Cấu hình huấn luyện PyTorch/SentenceTransformers
EPOCHS = 100
BATCH_SIZE = 16  # có thể tăng lên 32 nếu GPU đủ VRAM
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MODEL_NAME = 'BAAI/bge-m3'
OUTPUT_DIR = MYDRIVE / 'eform_btp/Models/EFormExcelMapper-v1'
print({'epochs':EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'model':MODEL_NAME})

# Chỉ bắt đầu khi có nhãn thật. Không dùng candidate chưa verified để train.
if len(train_rows) < 20:
    print('DỪNG AN TOÀN: cần ít nhất 20 train rows có verified=true và target_data_field_id.')
else:
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    model = SentenceTransformer(MODEL_NAME)
    examples = [InputExample(texts=[r.get('query',''), r.get('positive','')]) for r in train_rows if r.get('query') and r.get('positive')]
    loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
    loss = losses.MultipleNegativesRankingLoss(model)
    model.fit(train_objectives=[(loader, loss)], epochs=EPOCHS, warmup_steps=max(1, int(len(loader)*EPOCHS*WARMUP_RATIO)), optimizer_params={'lr': LEARNING_RATE}, output_path=str(OUTPUT_DIR), show_progress_bar=True)
    print('saved:', OUTPUT_DIR)